<a href="https://colab.research.google.com/github/jojojojo-jo/LRP_analysis_resnet/blob/main/Food101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torchvision.datasets import Food101
from torch.utils.data import Subset

# 1. Load the test split exactly as shown in the docs
full_test_set = Food101(root="./data", split="test", download=True)

100%|██████████| 5.00G/5.00G [03:49<00:00, 21.7MB/s]


AttributeError: 'Food101' object has no attribute 'targets'

In [ ]:
# 1. Download and load the base dataset
full_test_set = Food101(root="./data", split="test", download=True)

# 2. Extract indices for our 3 target classes
target_names = ["steak", "sushi", "pizza"]
target_ids = [full_test_set.classes.index(name) for name in target_names]

# Find where the labels match our targets in the internal '_labels' list
filtered_indices = [
    idx for idx, label in enumerate(full_test_set._labels)
    if label in target_ids
]

# 3. Restructure it into a hard-filtered Subset
clean_dataset = Subset(full_test_set, filtered_indices)

# 4. DESTRUCTIVE CLEANUP: Rewrite metadata to drop the other 98 classes
# Create a new mapping where: pizza -> 0, steak -> 1, sushi -> 2 (or sequential index)
new_class_to_idx = {name: i for i, name in enumerate(target_names)}

# Override the parent references inside your subset container
clean_dataset.classes = target_names
clean_dataset.class_to_idx = new_class_to_idx

# Map the internal labels inside the subset sequentially to [0, 1, 2]
# (Crucial so cross-entropy loss functions don't crash looking for index 92)
clean_dataset.targets = [
    new_class_to_idx[full_test_set.classes[full_test_set._labels[idx]]]
    for idx in filtered_indices
]

# Hook a dynamic override to custom getitem so it spits out the corrected 0-2 label IDs
def clean_getitem(idx):
    img, _ = full_test_set[filtered_indices[idx]]
    new_label = clean_dataset.targets[idx]
    return img, new_label

clean_dataset.__class__.__getitem__ = lambda self, idx: clean_getitem(idx)


# --- VERIFICATION ---
print(f"Total isolated images: {len(clean_dataset)}")  # 750
print(f"Available classes: {clean_dataset.classes}")     # ['steak', 'sushi', 'pizza']

# Check a sample to prove it gives labels inside [0, 1, 2] range
img, label = clean_dataset[0]
print(f"Sample 0 label index: {label} (Corresponds to: {clean_dataset.classes[label]})")

Total isolated images: 750
Available classes: ['steak', 'sushi', 'pizza']
Sample 0 label index: 0 (Corresponds to: steak)


In [ ]:
import os
import shutil

# 1. Define the path where torchvision extracts the images
# Adjust "./data" if you used a different root directory earlier
image_dir = "./data/food-101/images"

# 2. Specify the only 3 classes you want to keep
keep_classes = {"steak", "sushi", "pizza"}

# 3. Iterate through all folders and delete the rest from disk
if os.path.exists(image_dir):
    all_folders = os.listdir(image_dir)
    deleted_count = 0

    for folder in all_folders:
        if folder not in keep_classes:
            folder_path = os.path.join(image_dir, folder)
            if os.path.isdir(folder_path):
                shutil.rmtree(folder_path)  # Hard delete the directory
                deleted_count += 1

    print(f"Cleanup complete! Deleted {deleted_count} unwanted class folders.")
    print(f"Remaining folders in storage: {os.listdir(image_dir)}")
else:
    print(f"Error: The directory {image_dir} was not found. Check your dataset root path.")

Cleanup complete! Deleted 98 unwanted class folders.
Remaining folders in storage: ['steak', 'sushi', 'pizza']


In [ ]:
import os

# Define the base directory path
image_dir = "./data/food-101/images"
target_folders = ["pizza", "steak", "sushi"]

for folder in target_folders:
    folder_path = os.path.join(image_dir, folder)

    if os.path.exists(folder_path):
        # List all items and filter out any hidden system files if they exist
        images = [f for f in os.listdir(folder_path) if not f.startswith('.')]
        print(f"Folder '{folder}': {len(images)} images")
    else:
        print(f"Folder '{folder}' does not exist at the specified path.")

Folder 'pizza': 1000 images
Folder 'steak': 1000 images
Folder 'sushi': 1000 images


In [ ]:
import os
import shutil

# Paths based on default Torchvision extraction
base_dir = "./data/food-101"
images_dir = os.path.join(base_dir, "images")
meta_dir = os.path.join(base_dir, "meta")

# Target folders we want to build
train_root = "./data/train"
test_root = "./data/test"
target_classes = ["pizza", "steak", "sushi"]

# 1. Read the official test.txt file (Fixed file mode here!)
test_meta_path = os.path.join(meta_dir, "test.txt")
with open(test_meta_path, "r") as f:
    test_images = set(line.strip() for line in f.readlines())

# 2. Create clean destination directories
for cls in target_classes:
    os.makedirs(os.path.join(test_root, cls), exist_ok=True)

# 3. Physically move the test images
for cls in target_classes:
    class_dir = os.path.join(images_dir, cls)
    if not os.path.exists(class_dir):
        continue

    for img_name in os.listdir(class_dir):
        source_path = os.path.join(class_dir, img_name)
        relative_id = f"{cls}/{os.path.splitext(img_name)[0]}"

        if relative_id in test_images:
            dest_path = os.path.join(test_root, cls, img_name)
            shutil.move(source_path, dest_path)

# 4. Delete the original directory containing everything else
shutil.rmtree(images_dir)

# --- VERIFICATION ---
print("Physical separation complete!")
for cls in target_classes:
    test_count = len(os.listdir(os.path.join(test_root, cls)))
    print(f"New Clean Folder 'test/{cls}': {test_count} images")

Physical separation complete!
New Clean Folder 'test/pizza': 250 images
New Clean Folder 'test/steak': 250 images
New Clean Folder 'test/sushi': 250 images
